# Stress Prediction v25 — 3-Model Ensemble

## Why v25
v24 (LB **0.388**) reached its ceiling because LightGBM alone has correlated errors. The OOF/LB gap (0.896 vs 0.388 = 0.51) confirms standard CV is heavily within-subject leaked — so threshold tuning on OOF doesn't transfer. The remaining lift comes from **better raw probabilities**, not better post-processing.

## What changes vs v24
| Component | v24 | v25 |
|---|---|---|
| Features | 166 (HRV time/freq, EDA peaks/tonic/phasic, ref-dev, etc.) | **same** |
| SMOTE class 1 (3×) | yes | **same** |
| Class weights uncapped | yes | **same** |
| Models | LightGBM × 10 seeds | **LGBM × 5 + XGBoost × 5 + CatBoost × 5** |
| Probability source | LGBM only | **mean(LGBM, XGB, CB)** |
| Submission post-processing | α=0.8, argmax, smooth=0.2 | **same proven config** |
| OOF diagnostic | StratifiedKFold (leaky, BA 0.896) | **GroupKFold (honest, expect ~0.45)** |

## What stays exactly the same
The proven post-processing (`α=0.8`, argmax, `smooth=0.2`) is locked in because v24 showed it wins by distribution match. We're not redoing that search — we're feeding it better probabilities.

## Outputs (3 submissions, in order of preference)
1. `submission_ensemble.csv` — **3-model average** + α=0.8 argmax (primary bet)
2. `submission_lgbm_only.csv` — LGBM only with same post-proc (safety net, should reproduce ~0.388)
3. `submission_ensemble_t1soft.csv` — ensemble + tiny class-1 boost (t1=0.20) (small variant)


In [1]:
%pip -q install lightgbm xgboost catboost scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats
from scipy import signal as sps
from scipy.integrate import trapezoid

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, GroupKFold

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR    = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)

Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)


In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda']         = out['eda'].clip(0, 60)
    out['heart_rate']  = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id']        = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid']       = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress']    = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA  = clean_sensor(TRAIN_DATA)
TEST_DATA   = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL  = clean_label(TEST_LABEL)
print('Cleaned.')

Cleaned.


In [4]:
def compute_resting_baselines(sensor_df, low_pct=10):
    refs = {}
    for pid, grp in sensor_df.groupby('pid'):
        hr   = grp['heart_rate'].values.astype(float)
        eda  = grp['eda'].values.astype(float)
        temp = grp['temperature'].values.astype(float)
        valid = np.isfinite(hr) & np.isfinite(eda)
        if valid.sum() < 100:
            refs[pid] = {'hr': float(np.nanmedian(hr)) if valid.any() else 70.0,
                         'eda': float(np.nanmedian(eda)) if valid.any() else 1.0,
                         'temp': float(np.nanmedian(temp)) if valid.any() else 33.0,
                         'hr_std': 5.0, 'eda_std': 0.5,
                         'hr_median': float(np.nanmedian(hr)) if valid.any() else 70.0}
            continue
        hr_v, eda_v, temp_v = hr[valid], eda[valid], temp[valid]
        hr_z  = (hr_v  - hr_v.mean())  / (hr_v.std()  + 1e-9)
        eda_z = (eda_v - eda_v.mean()) / (eda_v.std() + 1e-9)
        arousal  = hr_z + eda_z
        thr      = np.percentile(arousal, low_pct)
        rest_mask = arousal < thr
        if rest_mask.sum() < 10:
            rest_mask = np.ones(len(arousal), dtype=bool)
        refs[pid] = {
            'hr':        float(np.median(hr_v[rest_mask])),
            'eda':       float(np.median(eda_v[rest_mask])),
            'temp':      float(np.median(temp_v[rest_mask])),
            'hr_std':    float(np.std(hr_v[rest_mask]) + 1e-3),
            'eda_std':   float(np.std(eda_v[rest_mask]) + 1e-3),
            'hr_median': float(np.median(hr_v)),
        }
    return refs

TRAIN_REFS = compute_resting_baselines(TRAIN_DATA)
TEST_REFS  = compute_resting_baselines(TEST_DATA)
print('Baselines computed. Train:', len(TRAIN_REFS), '| Test:', len(TEST_REFS))

Baselines computed. Train: 7 | Test: 8


## Feature Extraction (166 features — same as v23/v24)

In [5]:
WINDOW_MS = 180_000
HALF_MS   = 90_000
THIRD_MS  = 60_000
SHORT_MS  = 60_000
LONG_MS   = 300_000
XLONG_MS  = 600_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']: f['hrv_'+k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff) else 0.0
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff) > 25))*100 if len(rr_diff) else 0.0
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff) > 50))*100 if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f

def hrv_frequency_domain(bpm_series):
    f = {'hrv_vlf':np.nan,'hrv_lf':np.nan,'hrv_hf':np.nan,'hrv_lf_hf':np.nan,'hrv_total_power':np.nan}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 60: return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    if len(bpm_1hz) < 30: return f
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_c = rr - rr.mean()
    nperseg = min(len(rr_c), 64)
    if nperseg < 16: return f
    try:
        freqs, psd = sps.welch(rr_c, fs=1.0, nperseg=nperseg, noverlap=nperseg//2, scaling='density')
        def bp(lo, hi):
            mask = (freqs >= lo) & (freqs < hi)
            return float(trapezoid(psd[mask], freqs[mask])) if mask.sum() >= 2 else 0.0
        f['hrv_vlf'] = bp(0.0033, 0.04); f['hrv_lf'] = bp(0.04, 0.15); f['hrv_hf'] = bp(0.15, 0.40)
        f['hrv_total_power'] = f['hrv_vlf'] + f['hrv_lf'] + f['hrv_hf']
        f['hrv_lf_hf'] = f['hrv_lf'] / (f['hrv_hf'] + 1e-6)
    except: pass
    return f

def eda_peak_features(eda_series):
    f = {'eda_n_peaks':np.nan,'eda_peaks_per_min':np.nan,'eda_mean_prominence':np.nan,
         'eda_max_prominence':np.nan,'eda_mean_width':np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40: return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 16: return f
    try:
        wl = min(len(eda_4hz) - (1 if len(eda_4hz)%2==0 else 0), 15)
        if wl < 5: wl = 5
        if wl % 2 == 0: wl -= 1
        trend = sps.savgol_filter(eda_4hz, window_length=wl, polyorder=2) if len(eda_4hz)>20 else eda_4hz
        phasic = eda_4hz - trend + np.mean(eda_4hz)
        peaks, props = sps.find_peaks(phasic, prominence=0.02, distance=4, width=1)
        f['eda_n_peaks'] = float(len(peaks))
        dur_min = len(eda_4hz)/(4.0*60)
        f['eda_peaks_per_min'] = float(len(peaks)/dur_min) if dur_min > 0 else 0.0
        if len(peaks) > 0:
            f['eda_mean_prominence'] = float(np.mean(props['prominences']))
            f['eda_max_prominence']  = float(np.max(props['prominences']))
            f['eda_mean_width']      = float(np.mean(props['widths']))
        else:
            f['eda_mean_prominence'] = f['eda_max_prominence'] = f['eda_mean_width'] = 0.0
    except: pass
    return f

def eda_tonic_phasic_features(eda_series):
    f = {'eda_tonic_mean':np.nan,'eda_tonic_std':np.nan,'eda_phasic_mean':np.nan,
         'eda_phasic_std':np.nan,'eda_phasic_energy':np.nan,'eda_phasic_max':np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40: return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 20: return f
    try:
        wlen = min(len(eda_4hz) - (1 if len(eda_4hz)%2==0 else 0), 61)
        if wlen < 5: wlen = 5
        if wlen % 2 == 0: wlen -= 1
        tonic  = sps.savgol_filter(eda_4hz, window_length=wlen, polyorder=1)
        phasic = eda_4hz - tonic
        phasic_pos = np.maximum(phasic, 0)
        f['eda_tonic_mean']    = float(np.mean(tonic))
        f['eda_tonic_std']     = float(np.std(tonic))
        f['eda_phasic_mean']   = float(np.mean(phasic_pos))
        f['eda_phasic_std']    = float(np.std(phasic))
        f['eda_phasic_energy'] = float(np.sum(phasic_pos**2))
        f['eda_phasic_max']    = float(np.max(phasic_pos)) if len(phasic_pos) else 0.0
    except: pass
    return f

def accel_jerk_features(ax, ay, az):
    f = {'jerk_mean':np.nan,'jerk_std':np.nan,'jerk_max':np.nan}
    if len(ax) < 5: return f
    try:
        mag = np.sqrt(ax**2+ay**2+az**2)
        jerk = np.abs(np.diff(mag))
        f['jerk_mean'] = float(np.mean(jerk))
        f['jerk_std']  = float(np.std(jerk))
        f['jerk_max']  = float(np.max(jerk))
    except: pass
    return f

def timestamp_features(ts):
    sec = ts / 1000.0
    hour_in_day = (sec / 3600.0) % 24
    return {
        'ts_hour_sin': float(np.sin(2*np.pi*hour_in_day/24)),
        'ts_hour_cos': float(np.cos(2*np.pi*hour_in_day/24)),
    }

In [6]:
def extract_features(label_df, sensor_df, pid_enc_map, refs):
    sensor_by_pid = {pid: g for pid, g in sensor_df.groupby('pid')}
    label_ts_by_pid = {pid: g['timestamp'].values for pid, g in label_df.groupby('pid')}
    rows = []
    for n, lr in enumerate(label_df.itertuples(index=False), 1):
        pid, ts, lid = lr.pid, lr.timestamp, lr.id
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values
        wa     = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts), SENSOR_COLS]
        wf     = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - HALF_MS),   SENSOR_COLS]
        wl     = sg.loc[(ta >= ts - HALF_MS)   & (ta <= ts),             SENSOR_COLS]
        wt1    = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - 2*THIRD_MS),SENSOR_COLS]
        wt3    = sg.loc[(ta >= ts - THIRD_MS)  & (ta <= ts),             SENSOR_COLS]
        wshort = sg.loc[(ta >= ts - SHORT_MS)  & (ta <= ts),             SENSOR_COLS]
        wlong  = sg.loc[(ta >= ts - LONG_MS)   & (ta <= ts),             SENSOR_COLS]
        wxlong = sg.loc[(ta >= ts - XLONG_MS)  & (ta <= ts),             SENSOR_COLS]

        feat['window_count'] = len(wa)
        feat['window_completeness'] = min(1.0, len(wa) / max(WINDOW_MS/1000, 1))

        for c in SENSOR_COLS:
            v   = wa[c].dropna().values.astype(float)
            vf  = wf[c].dropna().values.astype(float)
            vl  = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']    = float(np.mean(v))
            feat[f'{c}_std']     = float(np.std(v))
            feat[f'{c}_min']     = float(np.min(v))
            feat[f'{c}_max']     = float(np.max(v))
            feat[f'{c}_median']  = float(np.median(v))
            feat[f'{c}_skew']    = float(spstats.skew(v)) if len(v)>2 else 0.0
            feat[f'{c}_kurt']    = float(spstats.kurtosis(v)) if len(v)>2 else 0.0
            feat[f'{c}_range']   = float(np.max(v)-np.min(v))
            feat[f'{c}_q25']     = float(np.percentile(v,25))
            feat[f'{c}_q75']     = float(np.percentile(v,75))
            feat[f'{c}_iqr']     = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta']   = float(np.mean(vl)-np.mean(vf)) if len(vf) and len(vl) else 0.0
            feat[f'{c}_slope']   = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.0
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1']    = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan
        feat.update(accel_jerk_features(ax, ay, az))
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat.update(hrv_frequency_domain(wa['heart_rate']))
        feat.update(eda_peak_features(wa['eda']))
        feat.update(eda_tonic_phasic_features(wa['eda']))

        for c in ['heart_rate','eda']:
            vs = wshort[c].dropna().values.astype(float)
            if len(vs) == 0:
                feat[f'{c}_short_mean'] = feat[f'{c}_short_std'] = \
                feat[f'{c}_short_max']  = feat[f'{c}_short_slope'] = np.nan
                continue
            feat[f'{c}_short_mean']  = float(np.mean(vs))
            feat[f'{c}_short_std']   = float(np.std(vs))
            feat[f'{c}_short_max']   = float(np.max(vs))
            feat[f'{c}_short_slope'] = float(np.polyfit(np.linspace(0,1,len(vs)),vs,1)[0]) if len(vs)>2 else 0.0

        for c in ['heart_rate','eda','temperature']:
            vl2 = wlong[c].dropna().values.astype(float)
            if len(vl2) == 0:
                feat[f'{c}_long_mean'] = feat[f'{c}_long_std'] = feat[f'{c}_long_slope'] = np.nan
                continue
            feat[f'{c}_long_mean']  = float(np.mean(vl2))
            feat[f'{c}_long_std']   = float(np.std(vl2))
            feat[f'{c}_long_slope'] = float(np.polyfit(np.linspace(0,1,len(vl2)),vl2,1)[0]) if len(vl2)>2 else 0.0

        for c in ['heart_rate','eda']:
            v3 = wa[c].dropna().values.astype(float)
            v5 = wlong[c].dropna().values.astype(float)
            feat[f'{c}_3vs5min'] = float(np.mean(v3)-np.mean(v5)) if len(v3)>0 and len(v5)>0 else np.nan

        for c in ['temperature','heart_rate']:
            vxl = wxlong[c].dropna().values.astype(float)
            if len(vxl) > 2:
                feat[f'{c}_xlong_slope'] = float(np.polyfit(np.linspace(0,1,len(vxl)),vxl,1)[0])
                feat[f'{c}_xlong_mean']  = float(np.mean(vxl))
            else:
                feat[f'{c}_xlong_slope'] = feat[f'{c}_xlong_mean'] = np.nan

        ref = refs.get(pid, {})
        if ref:
            hr_m  = feat.get('heart_rate_mean', np.nan)
            eda_m = feat.get('eda_mean', np.nan)
            tmp_m = feat.get('temperature_mean', np.nan)
            feat['hr_dev_rest']      = (hr_m  - ref['hr'])  if np.isfinite(hr_m)  else np.nan
            feat['hr_dev_rest_std']  = (hr_m  - ref['hr'])  / ref['hr_std']  if np.isfinite(hr_m)  else np.nan
            feat['eda_dev_rest']     = (eda_m - ref['eda']) if np.isfinite(eda_m) else np.nan
            feat['eda_dev_rest_std'] = (eda_m - ref['eda']) / ref['eda_std'] if np.isfinite(eda_m) else np.nan
            feat['temp_dev_rest']    = (tmp_m - ref['temp']) if np.isfinite(tmp_m) else np.nan
            feat['compound_stress']  = feat['hr_dev_rest_std'] + feat['eda_dev_rest_std'] \
                                       if np.isfinite(feat['hr_dev_rest_std']) and np.isfinite(feat['eda_dev_rest_std']) else np.nan
            hr_med = ref.get('hr_median', ref['hr'])
            feat['hr_above_median'] = float(hr_m > hr_med) if np.isfinite(hr_m) else np.nan
            feat['hr_dev_median']   = (hr_m - hr_med)      if np.isfinite(hr_m) else np.nan
        else:
            for k in ['hr_dev_rest','hr_dev_rest_std','eda_dev_rest','eda_dev_rest_std',
                      'temp_dev_rest','compound_stress','hr_above_median','hr_dev_median']:
                feat[k] = np.nan

        try:
            hr_a  = wa['heart_rate'].dropna().values.astype(float)
            eda_a = wa['eda'].dropna().values.astype(float)
            tmp_a = wa['temperature'].dropna().values.astype(float)
            nm    = min(len(hr_a), len(eda_a), len(tmp_a))
            if nm >= 30:
                hr_a, eda_a, tmp_a = hr_a[:nm], eda_a[:nm], tmp_a[:nm]
                feat['corr_hr_eda']  = float(np.corrcoef(hr_a, eda_a)[0,1])  if hr_a.std()>1e-6 and eda_a.std()>1e-6  else 0.0
                feat['corr_hr_temp'] = float(np.corrcoef(hr_a, tmp_a)[0,1])  if hr_a.std()>1e-6 and tmp_a.std()>1e-6  else 0.0
                feat['corr_eda_temp']= float(np.corrcoef(eda_a,tmp_a)[0,1])  if eda_a.std()>1e-6 and tmp_a.std()>1e-6 else 0.0
            else:
                feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan
        except:
            feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan

        try:
            all_ts   = label_ts_by_pid.get(pid, np.array([ts]))
            prior_ts = all_ts[all_ts < ts]
            next_ts  = all_ts[all_ts > ts]
            feat['label_gap_prev_sec'] = float((ts - prior_ts[-1])/1000) if len(prior_ts)>0 else np.nan
            feat['label_gap_next_sec'] = float((next_ts[0]  - ts)/1000)  if len(next_ts)>0  else np.nan
        except:
            feat['label_gap_prev_sec'] = feat['label_gap_next_sec'] = np.nan

        feat.update(timestamp_features(ts))
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}
print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map, TRAIN_REFS)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL,  TEST_DATA,  train_pid_map, TEST_REFS)
print(f'train: {train_features.shape} | test: {test_features.shape}')

Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028
train: (815, 165) | test: (1028, 165)


In [7]:
tli = TRAIN_LABEL.set_index('id')
y   = tli.loc[train_features.index, 'stress'].astype(int)
groups = tli.loc[train_features.index, 'pid']  # for GroupKFold

common_cols    = [c for c in train_features.columns if c in test_features.columns]
train_features = train_features[common_cols]
test_features  = test_features[common_cols]

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features), columns=common_cols, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),      columns=common_cols, index=test_features.index)

counts = Counter(y); total = len(y)
print('Class dist:', dict(counts))

class_weights = {
    0: total / (3 * counts[0]),
    1: total / (3 * counts[1]),
    2: total / (3 * counts[2]),
}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior    = np.array([counts[i]/total for i in range(3)])
print('Class weights (uncapped):', {k: round(v,3) for k,v in class_weights.items()})
print('Train prior:', train_prior.round(3).tolist())

Class dist: {1: 66, 0: 162, 2: 587}
Class weights (uncapped): {0: 1.677, 1: 4.116, 2: 0.463}
Train prior: [0.199, 0.081, 0.72]


## SMOTE-style class 1 oversampling
Same as v24: duplicate class 1 samples 3× with small Gaussian noise. Also augment groups (PID labels) so GroupKFold can still split correctly when used on augmented data.

In [8]:
def smote_minority_with_groups(X, y, groups_arr, minority_class=1, n_copies=3, noise_std_frac=0.01, seed=42):
    rng = np.random.RandomState(seed)
    mask = (y == minority_class).values
    X_min     = X.values[mask]
    y_min     = y.values[mask]
    g_min     = groups_arr[mask]
    col_stds  = X.values.std(axis=0) * noise_std_frac
    
    X_parts = [X.values];   y_parts = [y.values];   g_parts = [groups_arr]
    for _ in range(n_copies):
        noise = rng.randn(*X_min.shape) * col_stds
        X_parts.append(X_min + noise)
        y_parts.append(y_min)
        g_parts.append(g_min)  # synthetic samples inherit original PID
    X_aug = np.vstack(X_parts)
    y_aug = np.concatenate(y_parts)
    g_aug = np.concatenate(g_parts)
    idx   = rng.permutation(len(y_aug))
    return (pd.DataFrame(X_aug[idx], columns=X.columns),
            pd.Series(y_aug[idx], name=y.name),
            g_aug[idx])

groups_arr = groups.values
X_aug, y_aug, groups_aug = smote_minority_with_groups(X_imp, y, groups_arr, minority_class=1, n_copies=3, noise_std_frac=0.01)

counts_aug = Counter(y_aug)
total_aug  = len(y_aug)
class_weights_aug = {k: total_aug / (3 * counts_aug[k]) for k in [0,1,2]}
sample_weights_aug = np.array([class_weights_aug[int(yi)] for yi in y_aug])

print(f'After SMOTE — class dist: {dict(counts_aug)}')
print(f'Weights (post-oversample): {[round(v,3) for v in class_weights_aug.values()]}')
print(f'Total training rows: {len(y_aug)}')

After SMOTE — class dist: {2: 587, 1: 264, 0: 162}
Weights (post-oversample): [2.084, 1.279, 0.575]
Total training rows: 1013


In [9]:
def make_session_groups(label_df, gap_ms=30*60*1000):
    labels = label_df.copy().reset_index(drop=True)
    labels['rowpos'] = np.arange(len(labels))
    out = []
    for pid, grp in labels.sort_values(['pid','timestamp']).groupby('pid', sort=False):
        ts   = grp['timestamp'].values.astype(float)
        sess = np.cumsum(np.r_[0, np.diff(ts) > gap_ms])
        for sid in np.unique(sess):
            out.append(grp['rowpos'].values[sess == sid])
    return out

def smooth_by_session(proba, sessions, strength=0.20):
    out = proba.copy()
    for idx in sessions:
        mean = proba[idx].mean(axis=0, keepdims=True)
        out[idx] = (1 - strength) * proba[idx] + strength * mean
    return out

TEST_SESSIONS = make_session_groups(TEST_LABEL)
print('Test sessions:', len(TEST_SESSIONS))

Test sessions: 106


## Phase 1 — GroupKFold OOF (HONEST cross-subject diagnostic)

Standard StratifiedKFold puts the same nurse in train and validation, leaking subject identity. v24's OOF BA was 0.896 because of this leak — but LB was 0.388, a 0.51 gap.

GroupKFold holds out one nurse at a time, exactly mimicking the test condition (8 unseen nurses). The OOF BA from this is much lower (~0.40–0.50) but **honest**. Use it to:
- See if the ensemble actually beats LightGBM cross-subject
- Get a realistic LB ceiling

We run only LightGBM here for speed; the diagnostic question is whether features + ensembling generalize, not which model is best alone.

In [10]:
LGBM_PARAMS = dict(
    n_estimators=1500, learning_rate=0.02, num_leaves=63, max_depth=-1,
    min_child_samples=20, subsample=0.6, subsample_freq=1, colsample_bytree=0.4,
    reg_alpha=0.3, reg_lambda=0.5, class_weight='balanced',
    objective='multiclass', num_class=3, n_jobs=-1, verbose=-1,
)
XGB_PARAMS = dict(
    n_estimators=2000, learning_rate=0.02, max_depth=6,
    subsample=0.7, colsample_bytree=0.5, reg_alpha=0.3, reg_lambda=0.5,
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    n_jobs=-1, verbosity=0, tree_method='hist',
)
CAT_PARAMS = dict(
    iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
    loss_function='MultiClass', verbose=False,
    allow_writing_files=False, thread_count=-1,
)

def fit_lgbm(X_tr, y_tr, X_val, y_val, sw_tr, seed):
    m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
    m.fit(X_tr, y_tr, sample_weight=sw_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)])
    return m

def fit_xgb(X_tr, y_tr, X_val, y_val, sw_tr, seed):
    m = xgb.XGBClassifier(
        **{**XGB_PARAMS, 'random_state': seed,
           'early_stopping_rounds': 150}
    )
    m.fit(X_tr, y_tr, sample_weight=sw_tr,
          eval_set=[(X_val, y_val)], verbose=False)
    return m

def fit_cat(X_tr, y_tr, X_val, y_val, sw_tr, seed):
    m = CatBoostClassifier(**{**CAT_PARAMS, 'random_seed': seed,
                              'early_stopping_rounds': 150})
    m.fit(X_tr, y_tr, sample_weight=sw_tr,
          eval_set=(X_val, y_val), verbose=False)
    return m

def proba_xgb(model, X):
    return model.predict_proba(X)

def proba_cat(model, X):
    p = model.predict_proba(X)
    return np.asarray(p)

# === Phase 1: GroupKFold OOF on ORIGINAL data — 3 models ===
print('=== Phase 1: GroupKFold OOF (per-PID hold-out, no augmentation) ===')
unique_pids = groups.unique()
n_groups = len(unique_pids)
print(f'  Holding out 1 of {n_groups} nurses at a time')

oof_lgbm = np.zeros((len(X_imp), 3))
oof_xgb  = np.zeros((len(X_imp), 3))
oof_cat  = np.zeros((len(X_imp), 3))

gkf = GroupKFold(n_splits=n_groups)
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_imp, y, groups), 1):
    held_pid = groups.iloc[val_idx[0]]
    y_val_unique = sorted(y.iloc[val_idx].unique())

    X_tr, X_val = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx],     y.iloc[val_idx]
    sw_tr = sample_weights[tr_idx]

    # need at least 2 classes in val for early stopping to be meaningful
    if len(set(y_tr)) < 3:
        # fallback: skip if missing classes in train
        continue

    m_l = fit_lgbm(X_tr, y_tr, X_val, y_val, sw_tr, seed=42)
    m_x = fit_xgb (X_tr, y_tr, X_val, y_val, sw_tr, seed=42)
    m_c = fit_cat (X_tr, y_tr, X_val, y_val, sw_tr, seed=42)
    oof_lgbm[val_idx] = m_l.predict_proba(X_val)
    oof_xgb [val_idx] = proba_xgb(m_x, X_val)
    oof_cat [val_idx] = proba_cat(m_c, X_val)

    p_ens = (oof_lgbm[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3
    ba_l = balanced_accuracy_score(y_val, oof_lgbm[val_idx].argmax(1)) if len(y_val_unique) >= 2 else float('nan')
    ba_x = balanced_accuracy_score(y_val, oof_xgb [val_idx].argmax(1)) if len(y_val_unique) >= 2 else float('nan')
    ba_c = balanced_accuracy_score(y_val, oof_cat [val_idx].argmax(1)) if len(y_val_unique) >= 2 else float('nan')
    ba_e = balanced_accuracy_score(y_val, p_ens.argmax(1))             if len(y_val_unique) >= 2 else float('nan')
    print(f'  hold {held_pid:>5s} (n={len(val_idx):3d}, classes={y_val_unique}): '
          f'LGBM={ba_l:.3f}  XGB={ba_x:.3f}  CAT={ba_c:.3f}  ENS={ba_e:.3f}')

# pool-wide BA on rows that had non-trivial validation
mask = oof_lgbm.sum(1) > 0
y_arr = y.values[mask]
ba_lgbm = balanced_accuracy_score(y_arr, oof_lgbm[mask].argmax(1))
ba_xgb  = balanced_accuracy_score(y_arr, oof_xgb[mask].argmax(1))
ba_cat  = balanced_accuracy_score(y_arr, oof_cat[mask].argmax(1))
oof_ens = (oof_lgbm + oof_xgb + oof_cat) / 3
ba_ens  = balanced_accuracy_score(y_arr, oof_ens[mask].argmax(1))
print(f'\n=== HONEST cross-subject GroupKFold OOF BA ===')
print(f'  LGBM only : {ba_lgbm:.4f}')
print(f'  XGB only  : {ba_xgb:.4f}')
print(f'  CAT only  : {ba_cat:.4f}')
print(f'  Ensemble  : {ba_ens:.4f}   <-- if > LGBM, ensembling generalizes')

=== Phase 1: GroupKFold OOF (per-PID hold-out, no augmentation) ===
  Holding out 1 of 7 nurses at a time
  hold  C8Q6 (n=152, classes=[np.int64(0), np.int64(2)]): LGBM=0.500  XGB=0.500  CAT=0.500  ENS=0.500
  hold  P4DZ (n=144, classes=[np.int64(0), np.int64(1), np.int64(2)]): LGBM=0.327  XGB=0.333  CAT=0.327  ENS=0.269
  hold  F1ZM (n=137, classes=[np.int64(1), np.int64(2)]): LGBM=0.481  XGB=0.489  CAT=0.485  ENS=0.485
  hold  HDS9 (n=135, classes=[np.int64(0), np.int64(2)]): LGBM=0.402  XGB=0.288  CAT=0.472  ENS=0.380
  hold  43JW (n= 93, classes=[np.int64(0), np.int64(2)]): LGBM=0.500  XGB=0.250  CAT=0.500  ENS=0.500
  hold  DT5C (n= 90, classes=[np.int64(0), np.int64(1), np.int64(2)]): LGBM=0.500  XGB=0.333  CAT=0.344  ENS=0.477
  hold  TPQI (n= 64, classes=[np.int64(0), np.int64(2)]): LGBM=0.638  XGB=0.671  CAT=0.707  ENS=0.719

=== HONEST cross-subject GroupKFold OOF BA ===
  LGBM only : 0.5963
  XGB only  : 0.2708
  CAT only  : 0.3056
  Ensemble  : 0.3409   <-- if > LGBM, ensem

## Phase 2 — Ensemble training on augmented data

Train LightGBM + XGBoost + CatBoost using StratifiedKFold on augmented data, 5 seeds each. Average probabilities across all 15 model groups.

Why StratifiedKFold here (not GroupKFold)? Phase 1 already gave us the honest cross-subject estimate. Phase 2 just needs stable test predictions, and StratifiedKFold gives more diverse training subsets per fold (which makes the model average more robust).

In [11]:
SEEDS    = [42, 7, 123, 17, 99]
N_SPLITS = 5

print('=== Phase 2: Ensemble training on augmented data ===')
print(f'  Models: LGBM + XGB + CAT  |  Seeds: {SEEDS}  |  Folds: {N_SPLITS}')

test_proba_lgbm = np.zeros((len(X_test_imp), 3))
test_proba_xgb  = np.zeros((len(X_test_imp), 3))
test_proba_cat  = np.zeros((len(X_test_imp), 3))
n_runs = 0

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_aug, y_aug), 1):
        X_tr, X_val = X_aug.iloc[tr_idx], X_aug.iloc[val_idx]
        y_tr, y_val = y_aug.iloc[tr_idx], y_aug.iloc[val_idx]
        sw_tr = sample_weights_aug[tr_idx]

        m_l = fit_lgbm(X_tr, y_tr, X_val, y_val, sw_tr, seed=seed)
        m_x = fit_xgb (X_tr, y_tr, X_val, y_val, sw_tr, seed=seed)
        m_c = fit_cat (X_tr, y_tr, X_val, y_val, sw_tr, seed=seed)

        test_proba_lgbm += m_l.predict_proba(X_test_imp)
        test_proba_xgb  += proba_xgb(m_x, X_test_imp)
        test_proba_cat  += proba_cat(m_c, X_test_imp)
        n_runs += 1
    print(f'  Seed {seed:4d} done  ({n_runs} model groups so far)')

test_proba_lgbm /= n_runs
test_proba_xgb  /= n_runs
test_proba_cat  /= n_runs
test_proba_ens   = (test_proba_lgbm + test_proba_xgb + test_proba_cat) / 3

print()
print('Raw test argmax dist (LGBM only):', dict(Counter(test_proba_lgbm.argmax(1))))
print('Raw test argmax dist (XGB only) :', dict(Counter(test_proba_xgb.argmax(1))))
print('Raw test argmax dist (CAT only) :', dict(Counter(test_proba_cat.argmax(1))))
print('Raw test argmax dist (ENSEMBLE) :', dict(Counter(test_proba_ens.argmax(1))))

=== Phase 2: Ensemble training on augmented data ===
  Models: LGBM + XGB + CAT  |  Seeds: [42, 7, 123, 17, 99]  |  Folds: 5
  Seed   42 done  (5 model groups so far)
  Seed    7 done  (10 model groups so far)
  Seed  123 done  (15 model groups so far)
  Seed   17 done  (20 model groups so far)
  Seed   99 done  (25 model groups so far)

Raw test argmax dist (LGBM only): {np.int64(2): 522, np.int64(0): 325, np.int64(1): 181}
Raw test argmax dist (XGB only) : {np.int64(2): 537, np.int64(0): 180, np.int64(1): 311}
Raw test argmax dist (CAT only) : {np.int64(2): 604, np.int64(1): 152, np.int64(0): 272}
Raw test argmax dist (ENSEMBLE) : {np.int64(2): 569, np.int64(0): 261, np.int64(1): 198}


## Submissions — proven post-processing

We use the v24-winning configuration (LB **0.388**): `α=0.8` prior calibration, `smooth=0.2` session smoothing, argmax decision. We're feeding this proven recipe **better probabilities** (3-model ensemble) instead of just LGBM.

In [12]:
def make_sub(proba, alpha, smooth_strength, t1, fname):
    cal = proba * (train_prior ** alpha)
    cal = cal / cal.sum(axis=1, keepdims=True)
    if smooth_strength > 0:
        cal = smooth_by_session(cal, TEST_SESSIONS, strength=smooth_strength)
    if t1 is None or t1 >= 0.33:
        preds = cal.argmax(1)
    else:
        preds = cal.argmax(1).copy()
        preds[cal[:, 1] >= t1] = 1
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds.astype(int)}).to_csv(fname, index=False)
    cnts = np.bincount(preds, minlength=3)
    fracs = cnts / len(preds)
    dev   = np.abs(fracs - train_prior).max()
    return preds, cnts, fracs, dev

# 1. PRIMARY: 3-model ensemble + proven post-proc
preds_e, cnts_e, fracs_e, dev_e = make_sub(test_proba_ens, alpha=0.8, smooth_strength=0.2, t1=None,
                                           fname='submission_ensemble.csv')
print(f'submission_ensemble.csv         dist={cnts_e.tolist()}  fracs={fracs_e.round(3).tolist()}  dev={dev_e:.3f}')

# 2. SAFETY: LGBM-only with same post-proc (expect ~0.388 — sanity check)
preds_l, cnts_l, fracs_l, dev_l = make_sub(test_proba_lgbm, alpha=0.8, smooth_strength=0.2, t1=None,
                                           fname='submission_lgbm_only.csv')
print(f'submission_lgbm_only.csv        dist={cnts_l.tolist()}  fracs={fracs_l.round(3).tolist()}  dev={dev_l:.3f}')

# 3. VARIANT: ensemble + small class-1 boost (t1=0.20 — only triggers if model is very confident class 1)
preds_t, cnts_t, fracs_t, dev_t = make_sub(test_proba_ens, alpha=0.8, smooth_strength=0.2, t1=0.20,
                                           fname='submission_ensemble_t1soft.csv')
print(f'submission_ensemble_t1soft.csv  dist={cnts_t.tolist()}  fracs={fracs_t.round(3).tolist()}  dev={dev_t:.3f}')

# Convenience: copy primary as 'submission.csv' too
import shutil
shutil.copy('submission_ensemble.csv', 'submission.csv')

print()
print('========== v25 SUMMARY ==========')
print(f'GroupKFold OOF BA (LGBM)     : {ba_lgbm:.4f}')
print(f'GroupKFold OOF BA (XGB)      : {ba_xgb:.4f}')
print(f'GroupKFold OOF BA (CAT)      : {ba_cat:.4f}')
print(f'GroupKFold OOF BA (ENSEMBLE) : {ba_ens:.4f}')
print()
print('SUBMISSION ORDER (primary → fallback):')
print('  1. submission_ensemble.csv          ← 3-model avg + α=0.8 + argmax')
print('  2. submission_ensemble_t1soft.csv   ← + small class-1 boost (variant)')
print('  3. submission_lgbm_only.csv         ← safety net, mirrors v24')
print('==================================')

submission_ensemble.csv         dist=[146, 69, 813]  fracs=[0.142, 0.067, 0.791]  dev=0.071
submission_lgbm_only.csv        dist=[191, 75, 762]  fracs=[0.186, 0.073, 0.741]  dev=0.021
submission_ensemble_t1soft.csv  dist=[112, 158, 758]  fracs=[0.109, 0.154, 0.737]  dev=0.090

========== v25 SUMMARY ==========
GroupKFold OOF BA (LGBM)     : 0.5963
GroupKFold OOF BA (XGB)      : 0.2708
GroupKFold OOF BA (CAT)      : 0.3056
GroupKFold OOF BA (ENSEMBLE) : 0.3409

SUBMISSION ORDER (primary → fallback):
  1. submission_ensemble.csv          ← 3-model avg + α=0.8 + argmax
  2. submission_ensemble_t1soft.csv   ← + small class-1 boost (variant)
  3. submission_lgbm_only.csv         ← safety net, mirrors v24
